## Importing libraries

In [1]:
import numpy as np
import itertools
import math
import copy

from numpy import kron as tensProd

## Supporting functions

In [2]:
def flatten_AndDetype(aList):
    '''
    Flattens the list and sets the types to all match
    '''
    return(np.array([item for sublist in aList for item in sublist]).tolist())

def convertBetweenEntVecOrderings(entVec, curOrdering, targOrdering):
    '''
    A function that will apply a permutation to an entropy vector, so that
    the ordering of the elements matches targOrdering, when the original 
    ordering was curOrdering
    '''

    permutation = [curOrdering.index(i) for i in targOrdering]

    return entVec[permutation]


def repeatedTensorProd(dMat, k):
    '''
    Applys a repeated tensor product of the input density matrix, k times
    '''
    if(k <= 0):
        return(np.matrix([[1]]))
    if(k == 1):
        return(dMat)
    else:
        targDenseMat = np.matrix([[1]])
        for i in range(k):
            targDenseMat = tensProd(targDenseMat, dMat)
        
        return(targDenseMat)

def checkIsValidEntVec_2N(v):
    '''
    Input:
        v: An np.array of non-negative integer values

    Output: 
        isValid: Either 0 or 1 (0 means it is not a valid entropy vector)
    '''

    if not (len(v) == 3):
        raise Exception("The length of v is: {}, when it should be 3".format(len(v)))
    
    a1 = v[1] + v[2] - v[0]
    a2 = v[0] + v[1] - v[2]
    a3 = v[0] + v[2] - v[1]

    if((a1 < 0) or (a2 < 0) or (a3 < 0)):
        isValid = 0
    elif((v[0] < 0) or (v[1] < 0) or (v[2] < 0)):
        isValid = 0
    else:
        isValid = 1

    return(isValid)


def reconstIntegerVec_2N(v):
    '''
    Input: 
        v: A vector of non-negative integer values

    Output:
        hatV: The reconstructed vector
        vCoefs: An array of length 4, which specifies the coefficients 
                of u1, u2, u3, u4 (in that order) that are required to
                reconstruct v. If it is not a valid entopy vector
                this will be all -1's
    '''


    isValidEntVec = checkIsValidEntVec_2N(v)

    if(not isValidEntVec):
        # If not a valid entropy vector then set it to be all -1's
        vCoefs = [-1,-1,-1,-1]
    else:
        vCoefs = [0,0,0,0]

        # Defining the vectors that will be our basis set
        u1 = np.array([0,1,1])
        u2 = np.array([1,0,1])
        u3 = np.array([1,1,0])
        u4 = np.array([1,1,1])

        uVec = [u1, u2, u3, u4]
        vecLabels = np.array([1,2,3])

        perm = np.argsort(v)[::-1]

        vprime = v[perm]
        newVecLabels = vecLabels[perm]

        u2prime = uVec[newVecLabels[1] - 1]
        u3prime = uVec[newVecLabels[2] - 1]

        b1 = vprime[1] + vprime[2] - vprime[0]
        b2 = vprime[0] - vprime[1]
        b3 = vprime[0] - vprime[2]

        hatV = (b1 * u4) + (b2 * u2prime) + (b3 * u3prime)

        vCoefs[3] = b1
        vCoefs[newVecLabels[0]-1] = 0
        vCoefs[newVecLabels[1]-1] = b2
        vCoefs[newVecLabels[2]-1] = b3
    
    return(hatV, vCoefs)
    

def vCoeffs2DensityOperator_2N(vCoeffs):
    '''
    Input: 
        vCoeffs: An array of entries specifying the coefficients of u1, u2, u3, u4 for the reconstruction

    Output:
        densMat: A density matrix that will have the same entropy vector as v
        bitPartitions: A list containing two lists that correspond to the indices of systems in A and B
    '''

    # u1 corresponds to the maximally entangled state
    u1DensMat = (1/2) * np.matrix([[1, 0, 0, 1], [0,0,0,0], [0,0,0,0], [1,0,0,1]])

    # u2 corresponds to the maximally mixed state on B, and A being degenerate
    u2DenseMat = (1/2) * np.eye(2)

    # u3 corresponds to the maximally mixed state on A, and B being degenerate
    u3DenseMat = (1/2) * np.eye(2)

    # u4 corresponds to a A and B being entirely classically correlated
    u4DenseMat = (1/2) * np.matrix([[1,0,0,0], [0,0,0,0], [0,0,0,0], [0,0,0,1]])

    # Initializing the density matrix that we will use
    densMat = np.matrix([[1]])

    # Arrays that will track the bits in system A and system B
    aBits = []
    bBits = []

    curBitCount = 0

    if(vCoeffs[0] > 0):
        u1DensRepeated = repeatedTensorProd(dMat=u1DensMat, k=vCoeffs[0])
        densMat = tensProd(densMat, u1DensRepeated)

        # Updating the labels of the A systems and B systems based on the tensor products
        aBits.append([curBitCount + 2*i for i in range(vCoeffs[0])])
        bBits.append([curBitCount + 2*i + 1 for i in range(vCoeffs[0])])
        curBitCount = curBitCount + (2*vCoeffs[0])

    if(vCoeffs[1] > 0):
        u2DensRepeated = repeatedTensorProd(dMat=u2DenseMat, k=vCoeffs[1])
        densMat = tensProd(densMat, u2DensRepeated)
        # Since u2 corresponds to a state on B, but A being degenerate, we only need to update B
        bBits.append([curBitCount + i for i in range(vCoeffs[1])])
        curBitCount = curBitCount + vCoeffs[1]

    if(vCoeffs[2] > 0):
        u3DensRepeated = repeatedTensorProd(dMat=u3DenseMat, k=vCoeffs[2])
        densMat = tensProd(densMat, u3DensRepeated)
        # Since u3 corresponds to a state on A, but B being degenerate, we only need to update A
        aBits.append([curBitCount + i for i in range(vCoeffs[2])])
        curBitCount = curBitCount + vCoeffs[2]
    
    if(vCoeffs[3] > 0):
        u4DensRepeated = repeatedTensorProd(dMat=u4DenseMat, k=vCoeffs[3])
        densMat = tensProd(densMat, u4DensRepeated)

        # Updating the labels of the A systems and B systems based on the tensor products
        aBits.append([curBitCount + 2*i for i in range(vCoeffs[3])])
        bBits.append([curBitCount + 2*i + 1 for i in range(vCoeffs[3])])
        curBitCount = curBitCount + (2*vCoeffs[3])
    
    aBitsFlat = flatten_AndDetype(aBits)
    bBitsFlat = flatten_AndDetype(bBits)

    bitPartitions = [aBitsFlat, bBitsFlat]

    return(densMat, bitPartitions)



def entVec2DenseMat_Int2N(entVec, entVecOrdering):
    '''
    Function that maps a non-negative valued entropy vector and its ordering,
    into a density matrix with the exact same entropy vector
    '''

    if(not (len(entVec) == 3)):
        raise Exception("Entropy vector is of length: {}, but this function only works if it has length 3".format(len(entVec)))
    correctOrdering = [[0,1], 0, 1]

    reorderedEntVec = convertBetweenEntVecOrderings(entVec=entVec, curOrdering=entVecOrdering, targOrdering=correctOrdering)

    vHat, vCoeffs = reconstIntegerVec_2N(v=reorderedEntVec)

    if(not np.array_equal(vHat, reorderedEntVec)):
        raise Exception("The reordered entropy vector and the reconstructed entropy vector differ")
    

    denseMat, bitPartitions = vCoeffs2DensityOperator_2N(vCoeffs=vCoeffs)

    return(denseMat, bitPartitions)

class entropyVector_MixedState:
    """
    Entropy vector class to make things easier to work with 
    (specifically to make labelling the elements of the entropy vector
    easier)
    """

    def __init__(self, densityMat, bitPartitions):

        '''
        :param densityMat: Density Matrix corresponding to the state
        :param partitionBits: List of lists that specify which bits belong to which partitions
        '''
        # Checking that the dimension of the density matrix 
        # will match the number of bits specified
        numBits = np.sum([len(targList) for targList in bitPartitions])
        if not (np.log2(densityMat.shape[0]) == numBits):
            
            raise Exception("Number of bits was {}, and the size of the density matrix was {}, " \
                            "which is incompatible".format(numBits, densityMat.shape[0]))
        
        entVec, subSystemCombs = compEntropyVec_MixedState(densityOp=densityMat, bitPartitions=bitPartitions, returnLabels=True)

        self.densityMatrix = densityMat
        self.subSystemLabels = bitPartitions
        self.entVec = entVec
        self.entVecLabels = subSystemCombs

class entropyVector_PureState:
    """
    Attributes
    ----------
    pureState : np.ndarray
        High dimensional pure state vector
    subSystemLabels : str
        Names for each of the subsystems
    entVec : np.ndarray
        Vector of entropy values for the subsystems
    entVecLabels: list[int]
        Combinations of sub-system labels corresponding to each entropy
    """
    def __init__(self, pureState, bitPartitions):
        """
        :param pureState: Vector representing the pure state
        :param partitionBits: List of lists that specify which bits belong to which partitions
        """
        
        # Checking that the dimension of the pure state 
        # matches the number of bits specified
        numBits = np.sum([len(targList) for targList in bitPartitions])
        if not (np.log2(pureState.shape[0]) == numBits):
            raise Exception("Number of bits was {}, and the size of the density matrix was {}, " \
                            "which is incompatible".format(numBits, pureState.shape[0]))
        
        # Normalizing the pure state (just in case)
        pureState = pureState / np.linalg.norm(pureState)

        entVec, subSystemCombs = compEntropyVec_PureState(pureState=pureState, bitPartitions=bitPartitions, returnLabels=True)
    
        self.pureState = pureState
        self.subSystemLabels = bitPartitions
        self.entVec = entVec
        self.entVecLabels = subSystemCombs

    
    

    def __add__(self, other):
        if not isinstance(other, entropyVector_PureState):
            raise Exception("Cannot add pure state entropy vector and {}".format(other))
        
        (combState, combLabels) = entVecAdd_PureState(self, other)
        
        return(entropyVector_PureState(pureState=combState, bitPartitions=combLabels))
            
    
    def __mul__(self, other):
        if not (isinstance(other, int)):
            raise Exception("Cannot multiply a pure state entropy vector and {}".format(other))
        if not (other > 0):
            raise Exception("Cannot multiply an entropy vector by {} (a non-positive integer)".format(other))
        
        (scaledState, scaledLabels) = entVecScalarMult_PureState(entVec=self, scalar=other)        
        return(entropyVector_PureState(pureState=scaledState, bitPartitions=scaledLabels))

    def __rmul__(self, other):
        if not (isinstance(other, int)):
            raise Exception("Cannot multiply a pure state entropy vector and {}".format(other))
        if not (other > 0):
            raise Exception("Cannot multiply an entropy vector by {} (a non-positive integer)".format(other))
        
        (scaledState, scaledLabels) = entVecScalarMult_PureState(entVec=self, scalar=other)        
        return(entropyVector_PureState(pureState=scaledState, bitPartitions=scaledLabels))

 
def entVecAdd_PureState_Reduc(entVec1_Store: tuple[list, np.ndarray], entVec2_Store: tuple[list, np.ndarray]):
    """
    A modified version of the below function that doesn't require all the properties
    of entropy vectors be present, i.e. we don't have to re-compute the entropy vector
    each time we add something
    
    :param entVec1_Store: A tuple with the first element being the labels of an entropy vector, and the second being the pure state
    :param entVec2_Store: A tuple with the first element being the labels of an entropy vector, and the second being the pure state
    """
    labelArray1 = bitmap2Array(entVec1_Store[0])
    labelArray2 = bitmap2Array(entVec2_Store[0])
    combinedLabels = array2Bitmap(labelArray1 + labelArray2)

    combinedPureState = np.kron(entVec1_Store[1], entVec2_Store[1])

    return(combinedPureState, combinedLabels)

def entVecAdd_PureState(entVec1: entropyVector_PureState, entVec2: entropyVector_PureState):
    labelArray1 = bitmap2Array(entVec1.subSystemLabels)
    labelArray2 = bitmap2Array(entVec2.subSystemLabels)
    combinedLabels = array2Bitmap(labelArray1 + labelArray2)

    combinedPureState = np.kron(entVec1.pureState, entVec2.pureState)
    
    return(combinedPureState, combinedLabels)

def entVecScalarMult_PureState(entVec: entropyVector_PureState, scalar: int):
    if not(scalar > 0):
        raise Exception("The scalar is {}, which is not > 0".format(scalar))

    origState = entVec.pureState
    origLabels = entVec.subSystemLabels
    origLabelArray = bitmap2Array(origLabels)
    updatedLabels = array2Bitmap(scalar * origLabelArray)

    curEntVec = copy.deepcopy(entVec)
    curState = curEntVec.pureState

    for i in range(scalar-1):
        curState = np.kron(curState, origState)

    return(curState, updatedLabels)
    

def compEntropyVec_PureState(pureState, bitPartitions, returnLabels=False):

    numSubSystems = len(bitPartitions)
    subSystemLabels = list(range(numSubSystems))
    # Here we are counting less combinations than the total possible
    # number of combinations required, because of symmetries
    # in the entropies of the reduced density matrices of
    # pure states
    numCombs = (2**(numSubSystems-1)) - 1
    systemCombs = [[] for _ in range( numCombs )]
    systemCombIndex = 0

    c_floor = math.floor((numSubSystems - 1) / 2)

    systemCombIndex = 0

    for combSize in range(c_floor):
        for subset in itertools.combinations(subSystemLabels, combSize+1):
            systemCombs[systemCombIndex] = list(subset)
            systemCombIndex = systemCombIndex + 1

    if(numSubSystems % 2 == 0):
        # Need to do a special computation for the even N's, 
        # because for the odd number, the subsystems of size
        # [up to] (N-1)/2 cover all the possible combinations
        # of subsystems on the other side of (N-1)/2, 
        # while the even ones want to have exactly half
        # of the subsystems of size N/2, since the other
        # half are redundant

        # An odd example:
        # N=3 and subsystems A,B,C
        # sA,sB,sC also cover sBC, sAC, sAB

        # An even example: 
        # N=4 and subsystems A,B,C,D.
        # sA, sB, sC, sD also cover sBCD, sACD, sABD, sABC
        # sAB, sAC, sAD also cover sCD, sBD, sBC

        # Therefore, we only need to generate the 
        # combinations of size N/2, but for a specific
        # system fixed to be in the combinations, since
        # this will cover all the ones with that system
        # not in the combinations (i.e. the other half)

        # Arbitrarily we fix the first system, but
        # any other system could be fixed instead
        nonFixedSubSys = subSystemLabels[1:]

        for unFixedSubset in itertools.combinations(nonFixedSubSys, c_floor):
            targComb = [0] + list(unFixedSubset)
            systemCombs[systemCombIndex] = targComb
            systemCombIndex = systemCombIndex + 1

    ## Computing the entropy vector

    # Generating an appropriate density operator
    if(isinstance(pureState, np.ndarray)):
        # If the pure state is an np.array, then
        # we convert it to a matrix to make things easier
        pureStateMat = np.matrix(pureState)
    else:
        pureStateMat = pureState

    densityOperator = pureStateMat @ pureStateMat.H

    entropyVec = np.full(shape=(numCombs,), fill_value=np.nan)
    for targCombInd in range(numCombs):
        targComb = systemCombs[targCombInd]

        systemsToTraceOut = list(set(subSystemLabels) - set(targComb))

        bitsToTraceOut = []
        for targSystem in systemsToTraceOut:
            bitsToTraceOut = bitsToTraceOut + bitPartitions[targSystem]
        if (targComb ==  subSystemLabels):
            subSystemEntropy = vnEntropy(densityMat=densityOperator)
        else:
            reducedSystem = partialTrace_TargBits(targDensityMat=densityOperator, bitsToTraceOut=bitsToTraceOut, indexing=0)

            subSystemEntropy = vnEntropy(densityMat=reducedSystem)

        entropyVec[targCombInd] = subSystemEntropy
    
    if(returnLabels):
        return(entropyVec, systemCombs)
    else:
        return(entropyVec)


def compEntropyVec_MixedState(densityOp, bitPartitions, returnLabels=False):
    ## Generating the possible pairings
    numSubSystems = len(bitPartitions)
    subSystemLabels = list(range(numSubSystems))
    numCombs = (2**numSubSystems) - 1

    systemCombs = [[] for _ in range( numCombs )]
    systemCombIndex = 0
    # Iteration code taken from: 
    # https://stackoverflow.com/questions/464864/how-to-get-all-possible-2n-combinations-of-a-list-s-elements-of-any-length
    for combSize in range(numSubSystems):
        for subset in itertools.combinations(subSystemLabels, combSize+1):
            systemCombs[systemCombIndex] = list(subset)
            systemCombIndex = systemCombIndex + 1
    
    ## Computing the entropy vector
    entropyVec = np.full(shape=(numCombs,), fill_value=np.nan)
    for targCombInd in range(numCombs):
        targComb = systemCombs[targCombInd]

        systemsToTraceOut = list(set(subSystemLabels) - set(targComb))

        bitsToTraceOut = []
        for targSystem in systemsToTraceOut:
            bitsToTraceOut = bitsToTraceOut + bitPartitions[targSystem]
        if (targComb ==  subSystemLabels):
            subSystemEntropy = vnEntropy(densityMat=densityOp)
        else:
            reducedSystem = partialTrace_TargBits(targDensityMat=densityOp, bitsToTraceOut=bitsToTraceOut, indexing=0)

            subSystemEntropy = vnEntropy(densityMat=reducedSystem)

        entropyVec[targCombInd] = subSystemEntropy
    if(returnLabels):
        return(entropyVec, systemCombs)
    else:
        return(entropyVec)

def bitmap2Array(subSystemBitmap: list[list[int]], subSystemLabels=None):
    """
    Converts the bitmap that is our standard method of assigning 
    qubits to a system, into an array that has the appropriate label
    at each of the indices
    
    :param subSystemBitmap: List of lists that associates bits to
                            subsystems

    Examples: 
        >>> subSystemBitmap1 = [[0,1], [2,3], [4,5]]
        >>> subSysStr1 = subSysBitmap2Str(subSystemBitmap1)
        >>> print(subSysStr1)
        Output: [0,0,1,1,2,2]

        >>> subSystemBitmap2 = [[0,1], [3,5], [2,4]]
        >>> subSysStr2 = subSysBitmap2Str(subSystemBitmap2)
        >>> print(subSysStr2)
        Output: [0,0,2,1,2,1]
    """
    numSubSystems = len(subSystemBitmap)

    if(subSystemLabels == None):
        subSystemLabels_Usable = list(range(numSubSystems))
    else:
        subSystemLabels_Usable = subSystemLabels

    numBits = np.sum([len(targList) for targList in subSystemBitmap])


    subSystem_OrderedArray = ["a" for _ in range(numBits)]
    for targSubSystemLabel in subSystemLabels_Usable:
        for targIndex in subSystemBitmap[targSubSystemLabel]:
            subSystem_OrderedArray[targIndex] = str(targSubSystemLabel)
    
    return(subSystem_OrderedArray)

def array2Bitmap(bitmapArray: list[str]):
    """
    Reverse function for bitmap2Array
    
    :param bitmapArray: Array of values that correspond to the subsystem of
                        each bit
    :type bitmapArray: list[str]
    """

    
    subSystemLabels = []
    for targBitLabel in bitmapArray:
        if not (targBitLabel in subSystemLabels):
            subSystemLabels.append(targBitLabel)
        
    numSubSystems = len(subSystemLabels)
    subSystemBitmap = [[] for _ in range(numSubSystems)]

    for targInd in range(len(bitmapArray)):
        targBitLabel = bitmapArray[targInd]
        subSysLabelIndex = subSystemLabels.index(targBitLabel)

        updatedBitmapArray = subSystemBitmap[subSysLabelIndex]
        updatedBitmapArray.append(targInd)
        subSystemBitmap[subSysLabelIndex] = updatedBitmapArray

    return(subSystemBitmap)





def gen2PartiteState(S, iterations=50):
    """
    Algorithm that will generate a pure state which can be decomposed into
    two pieces so that S(ρ_A) =  S(ρ_B) = S

    Additionally, this algorithm will produce a pure state in the lowest dimension
    that the entropy can be achieved

    :param S: von Neuman entropy of the pure state
    """

    ## Our first goal is to construct a density operator with the 
    ## specified entropy, so that we just need to purify it
    d = math.ceil(2**S)
    if(d == 1):
        # Solving for μ immeadiately, because we don't need λ's
        mu = newtonsMethod_EntropyFunctional(d=d, r=S)
        leftover = 1 - mu
        eigVals = np.array([mu, leftover])
    elif(d > 1):
        lambdaVals = 1/d
        # Computing the remainder term
        r = S - ((d-2) * h(lambdaVals))

        # Solving for the value of μ that will cause
        # the entropy of the state to most closely match S
        mu = newtonsMethod_EntropyFunctional(d=d, r=r)
        leftover = (2/d) - mu

        eigVals = np.zeros((d,))
        eigVals[0:-2] = lambdaVals
        eigVals[-2] = mu
        eigVals[-1] = leftover

    densityMat = np.diag(eigVals)

    pureStateVec = purifyMixedState(densityMat=densityMat)

    return(pureStateVec)

import numpy as np
import sympy as sp
from sympy.physics.quantum import TensorProduct as spTensProd
import scipy as scp

import math

## Utility functions to use for the project

def genRandPureState(dim, rngSeed=-1):
    if(rngSeed<= 0):
        rng = np.random.default_rng()
    else:
        rng = np.random.default_rng(seed=rngSeed)

    cmplxGaussSamp = rng.normal(loc=0, scale=1, size=(dim,1)) + 1j*rng.normal(loc=0, scale=1, size=(dim, 1))

    normalizedCmplxGauss = cmplxGaussSamp / np.linalg.norm(cmplxGaussSamp, ord=2)

    return(normalizedCmplxGauss)


def genPureStateDMats(dim, num=1, rngSeed=-1):
    pureStateDMats = np.full(shape=(dim, dim, num), fill_value=np.nan, dtype=np.complex128)

    for pStInd in range(num):
        if(rngSeed <= 0):
            targPSt = genRandPureState(dim=dim, rngSeed=rngSeed)
        else:
            targPSt = genRandPureState(dim=dim, rngSeed=rngSeed+pStInd)

        targPStDMat = np.matrix(targPSt) @ np.matrix(targPSt).H
        pureStateDMats[:,:,pStInd] = targPStDMat

    return(pureStateDMats)

def genRandDensityMat(dim, rngSeed=-1):

    # Setting up the random number generation
    if(rngSeed <= 0):
        rng = np.random.default_rng()
    else:
        rng = np.random.default_rng(seed=rngSeed)


    # Generating N samples from a N dimensional complex Gaussian to use for the Haar basis
    complexGaussianSamples = rng.normal(loc=0, scale=1, size=(dim, dim)) + 1j*rng.normal(loc=0, scale=1, size=(dim, dim))
    
    normalizedCmplxSamps = complexGaussianSamples / np.linalg.norm(complexGaussianSamples, axis=0)

    basisVectors = gram_schmidt(A=normalizedCmplxSamps)

    dirichletSample = rng.dirichlet(alpha=np.ones(shape=(dim,)), size=(1,))

    randDensityMat = np.zeros(shape=(dim, dim), dtype=basisVectors.dtype)

    for basisIndex in range(basisVectors.shape[0]):
        targBasisVec = np.matrix(basisVectors[:,basisIndex]).H
        targProb = dirichletSample[0,basisIndex]
        targRankOneMat = targBasisVec @ targBasisVec.H
        randDensityMat = randDensityMat + targProb * targRankOneMat

    return randDensityMat

# Inspired by: https://www.sfu.ca/~jtmulhol/py4math/linalg/np-gramschmidt/
def gram_schmidt(A):
    '''
    input: A set of linearly independent vectors stored
              as the columns of matrix A
       outpt: An orthongonal basis for the column space of A
       '''
    # get the number of vectors.
    
    n = A.shape[1]
    basis = np.full(shape=(n,n), fill_value=np.nan, dtype=A.dtype)

    for ind in range(n):
        if(ind == 0):
            # First vector we normalize and then accept
            basis[:,ind] = A[:,0] / np.linalg.norm(A[:,0])

        else:
            curVec = A[:,ind]
            for backwardsInd in range(ind):

                intermediateVar = (basis[:,backwardsInd] * np.dot(A[:,ind], np.conj(basis[:,backwardsInd])))

                curVec = curVec - intermediateVar
            curVec = curVec / np.linalg.norm(curVec)
            basis[:,ind] = curVec

    return(basis)


def genDensMats(dim, num=1, rngSeed=-1):
    densityMats = np.full(shape=(dim,dim, num), fill_value=np.nan, dtype=np.complex128)

    for densMatInd in range(num):
        if(rngSeed <= 0):
            targDenseMat = genRandDensityMat(dim=dim, rngSeed=rngSeed)
        else:
            # Incrementing the rng seed so the resulting density matrices aren't always the same
            targDenseMat = genRandDensityMat(dim=dim, rngSeed=rngSeed+densMatInd)

        densityMats[:,:, densMatInd] = targDenseMat

    return(densityMats)

def qudit(d, bit):
    """
    Generate the corresponding qudit in the d dimensional Hilbert space
    
    :param d: Qubit dimension of Hilbert space (i.e. dim(H) = 2**d)
    :param bit: Bit (e.g. bit=0 yields |00...00>)
    """
    hDim = 2**d
    if(bit > hDim):
        raise Exception("Bit number is too high. d: {}, bit: {}".format(d, bit))
    targQudit = np.zeros((hDim,1))
    targQudit[bit] = 1
    return(targQudit)

def h(x, base=2):
    """
    Entropy functional: h(x) = -x log_2(x)
    
    :param x:
    :param base: base of the logarithm
    """
    if(x in [0,1]):
        # By convention we take 0 log_2(0) = 0
        return 0
    elif(np.isclose(x, 0)):
        return 0
    else:
        return( -1*x*math.log(x, base) )


def vnEntropy(densityMat):

    eigVals, _ = np.linalg.eigh(densityMat)

    entropyTotal = 0
    for eigVal in eigVals:
        entropyTotal = entropyTotal + h(np.abs(eigVal))

    return(entropyTotal)


def intToBinStr(integer, length=-1):
    # Code for conversion to binary string and filling in leading bits is taken from:
    # https://stackoverflow.com/questions/73285087/how-can-i-convert-an-integer-to-its-binary-representation-as-a-numpy-array

    if(length<= 0):
        # If length not specified, then we just keep it min length
        binStr = bin(integer)[2:]
    else:
        binStr = bin(integer)[2:].zfill(length)
    
    return(binStr)
def binStrToInt(binStr):
    return( int(binStr, 2) )

def tensorProd(mat1, mat2, mode="np"):
    '''
    This is a function that should take as input two numpy matrices (or sympy matrices),
    and will return the tensor product (or kronecker product depending on your notation)
    '''

    # Deciding what kind of objects we're dealing with
    if(mode in ["numpy, np"]):
        modeName = "numpy"
    if(mode in ["sp", "sympy"]):
        modeName = "sympy"
    else:
        modeName = "numpy"

    
    if(modeName == "numpy"):
        # print("This is here in numpy mode")
        output = np.kron(mat1, mat2)
        
    else:
        output = spTensProd(mat1, mat2)
        
    return(output)

def partialTrace_TargBits(targDensityMat, bitsToTraceOut, mode="np", indexing=1):
    ''' 
    This is a function that takes as input a density matrix, and a set of bits of the 
    matrix that we would like to trace out, and then performs the partial trace of this density operator.
    IMPORTANT: This function assumes the bitsToTraceOut vector is using 1 indexed by default

    targDensityMat: should be a numpy or sympy (depending on mode variable) matrix
                    that we want to perform the partial trace of

    bitsToTraceOut: should be a list or tuple of integers that indicates
                    what indices we want to trace out 
                    (assumed to be 1 indexed)

    mode:           specifies whether the density matrix is a numpy or sympy matrix
    '''
    # Deciding what kind of objects we're dealing with
    if(mode in ["numpy, np"]):
        modeName = "numpy"
    if(mode in ["sp", "sympy"]):
        modeName = "sympy"
    else:
        modeName = "numpy"

    qubitDim = int(np.log2(targDensityMat.shape[0]))

    # Creating a list of dimensions that is 1 indexed for readability
    if(indexing == 1):
        listOfInds = np.arange(qubitDim) + 1
    elif(indexing == 0):
        listOfInds = np.arange(qubitDim)
    else:
        raise Exception("We only allow 0 and 1 indexing, but the indexing input was {}".format(indexing))

    d1 = len(bitsToTraceOut)

    d2 = qubitDim - d1

    # Pre-allocating the reduced density matrix
    if(modeName == "sympy"):
        reducedDensityMat = sp.Matrix(np.zeros((2**d2, 2**d2)))
    else:
        reducedDensityMat = np.zeros((2**d2, 2**d2))

    for binaryNumInd in range((2**d1)):

        targBinaryString = intToBinStr(binaryNumInd, length=d1)

        # Setting up a counter to track where in the binary string we are, 
        # as we iterate over all possible qubit indices
        removeCounter = 0
        
        # Pre-defining the left and right matrices we will be using in our
        # matrix multiplications
        # (sympy supports mat mul between np mats and sp mats, so just using np mat here)
        targLeftMat = np.matrix([[1]])
        targRightMat = np.matrix([[1]])
   

        for qubitSysInd in range(qubitDim):
            
            # If its a bit in the system that we want to remove, 
            # then we iterate through the binary vector, and 
            # select the appropriate vector based on the binary value
            if(listOfInds[qubitSysInd] in bitsToTraceOut):
                targBit = targBinaryString[removeCounter]
                removeCounter = removeCounter + 1
                # If the bit is 0, want to use <0| for left and |0> for right
                if(targBit == "0"):
                    curLeftMat = np.matrix([[1,0]], dtype=int)
                    curRightMat = np.matrix([[1],[0]], dtype=int)
                # If bit is 1, want to use <1| for left and |1> for right
                elif(targBit == "1"):
                    curLeftMat = np.matrix([[0,1]], dtype=int)
                    curRightMat = np.matrix([[0], [1]], dtype=int)
                else:
                    raise Exception("Something went very wrong that should not have gone wrong. targBit is: {}".format(targBit))
            
            # If its not a bit to remove, then use an identity matrix in the tensor product
            else:
                
                curLeftMat = np.eye(2, dtype=int)
                curRightMat = np.eye(2, dtype=int)
            

            targLeftMat = tensorProd(targLeftMat, curLeftMat, mode="np")
            targRightMat = tensorProd(targRightMat, curRightMat, mode="np")


        # Performing the left and right multiplications
        curMatMultResult = targLeftMat @ targDensityMat @ targRightMat

        reducedDensityMat = reducedDensityMat + curMatMultResult


    return(reducedDensityMat)


def purifyMixedState(densityMat):


    d = densityMat.shape[0]

    eigVals, eigVecs = np.linalg.eigh(np.matrix(densityMat))

    targPureState = np.zeros(shape=(d**2,1), dtype=complex)

    for eInd in range(eigVals.shape[0]):
        targEigVal = eigVals[eInd]
        targEigVec = eigVecs[:,eInd]

        doubledEVec = math.sqrt(targEigVal) * np.kron(targEigVec, targEigVec)

        targPureState = targPureState + doubledEVec

    return(targPureState)


def newtonsMethod_EntropyFunctional(d, r):
    """
    Nevermind, I got Newtons method working
    
    :param d: dimension
    :param r: remainder
    """
    entFunc = lambda x : h(x) + h((2/d) - x) - r
    entFuncDeriv = lambda x : math.log2((2/(d*x)) - 1)

    midpoint = 3 / (2*d)

    newtonResult = scp.optimize.newton(func=entFunc, x0=midpoint, fprime=entFuncDeriv)

    return(newtonResult)

def successiveApprox_EntropyFunctional(d, r, numSteps=100):
    """
    A solver for the entropy functional that we want. 
    (Originally tried Newtons method, but that had domain issues)
    This only works because the derivative is strictly negative
    so the solution in the interval should be unique

    :param d: dimension of problem
    :param r: remainder term (we are trying to match this)
    """

    entFunc = lambda x : h(x) + h((2/d) - x)

    curLb = 1/d
    curUb = 2/d

    i = 0

    while i < numSteps:
        midPoint_x = (curLb + curUb) / 2
        curOutput = entFunc(midPoint_x)

        if(curOutput > r):
            curLb = midPoint_x
        elif(curOutput < r):
            curUb = midPoint_x
        else:
            # This only happens if we get equality,
            # at which point we want to return that midpoint
            i = numSteps
        i = i + 1
    
    return(midPoint_x)



## Code to generate a density operator and pure state corresponding to the input entropy vector

This will only work for a entropy vector of length 3, who's entries are non-negative integers

In [ ]:
inputEntVec = np.array([2,2,3])
curBitOrdering = [0, 1, [0,1]]

testDensMat, bitPartitions_MS = entVec2DenseMat_Int2N(entVec=inputEntVec, entVecOrdering=curBitOrdering)

# Computing the entropy vector for the generated density matrix to make sure it matches the 
# original entropy vector
entVec_MS = entropyVector_MixedState(densityMat=testDensMat, bitPartitions=bitPartitions_MS)
print("The entropy vector of the bipartite density matrix is: {}".format(entVec_MS.entVec))

# Constructing the pure state, and verifying that it has the same entropy vector as the input
testPureState = purifyMixedState(testDensMat)
numBits_MS = len(flatten_AndDetype(bitPartitions_MS))
pureStateBitInds = [i + numBits_MS for i in range(numBits_MS)]
bitPartitions_PS = bitPartitions_MS + [pureStateBitInds]
entVec_PS = entropyVector_PureState(pureState=testPureState, bitPartitions=bitPartitions_PS)
print("The entropy vector of the tripartite pure state is: {}".format(entVec_PS.entVec))


#### FOR JASON ####
#### Here is some example code for accessing properties of the entropy vectors ####
# entVec_PS is an object describing the pure state entropy vector
# entVec_PS.entVec: Returns the computed entropy vector
# entVec_PS.subSystemLabels: Returns the labels of the qubit subsystems
# entVec_PS.pureState: Returns the corresponding pure state
# entVec_PS.entVecLabels: Returns the labels of the entropy vector, saying which system each element corresponds to

The entropy vector of the bipartite density matrix is: [2. 2. 3.]
The entropy vector of the tripartite pure state is: [2. 2. 3.]
